In [2]:
# 🚌 APSRTC Bus Rush Prediction System
## Guntur & Tenali Routes — Complete ML Analysis
**Developer:** Gayatri Chukka | B.Tech CSE AI/ML | 2nd Year
**Model:** Random Forest | **Accuracy:** 80.5%
**Feature:** Passenger count between every stop pair

SyntaxError: invalid decimal literal (1081959698.py, line 3)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay
)
sns.set_style("whitegrid")
print("Libraries loaded ✅")

In [ ]:
df = pd.read_csv("apsrtc_bus_data.csv")
print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
df.head(10)

In [ ]:
## 2. Dataset Overview
The dataset has 8,000 rows and 11 columns.
Two key columns — max_segment_pax and min_segment_pax —
show passenger counts on the busiest and least crowded
stop-to-stop segments for each trip.

In [ ]:
print("Rush distribution:")
print(df["rush_level"].value_counts())
print("\nPassenger segment statistics:")
print(df[["passenger_count",
           "max_segment_pax",
           "min_segment_pax"]].describe().round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
COLORS = {"Low":"#27AE60","Medium":"#F39C12","High":"#E74C3C"}
rc   = df["rush_level"].value_counts()
bars = ax.bar(rc.index, rc.values,
              color=[COLORS[x] for x in rc.index],
              edgecolor="black", width=0.5)
ax.set_title("Rush Level Distribution — APSRTC Guntur & Tenali",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Rush Level"); ax.set_ylabel("Number of Trips")
for bar, val in zip(bars, rc.values):
    ax.text(bar.get_x()+bar.get_width()/2,
            val+30, str(val), ha="center", fontweight="bold")
plt.show()

In [ ]:
# Show passenger flow statistics by route
flow_analysis = df.groupby("route_id").agg(
    avg_passengers  = ("passenger_count","mean"),
    avg_max_segment = ("max_segment_pax","mean"),
    avg_min_segment = ("min_segment_pax","mean")
).round(1)
print("Passenger flow by route:")
print(flow_analysis.sort_values("avg_passengers", ascending=False))

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].hist(df["max_segment_pax"], bins=20,
             color="#E74C3C", alpha=0.8, edgecolor="black")
axes[0].set_title("Most Crowded Segment Distribution",
                   fontweight="bold")
axes[0].set_xlabel("Passengers")
axes[1].hist(df["min_segment_pax"], bins=20,
             color="#27AE60", alpha=0.8, edgecolor="black")
axes[1].set_title("Least Crowded Segment Distribution",
                   fontweight="bold")
axes[1].set_xlabel("Passengers")
plt.suptitle("Passenger Count Between Stops — Analysis",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
X_train = pd.read_csv("X_train.csv")
X_test  = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").squeeze()
y_test  = pd.read_csv("y_test.csv").squeeze()
model   = joblib.load("rush_model.pkl")
enc     = joblib.load("encoders.pkl")

y_pred  = model.predict(X_test)
acc     = accuracy_score(y_test, y_pred)
print(f"Random Forest Accuracy: {acc*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=enc["rush_level"].classes_))

In [ ]:
feat_imp = pd.DataFrame({
    "feature":    X_train.columns.tolist(),
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(9,5))
ax.barh(feat_imp["feature"], feat_imp["importance"],
        color="#2980B9", edgecolor="black")
ax.set_xlabel("Importance Score")
ax.set_title("Feature Importance — What Drives Bus Rush?",
             fontsize=13, fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop feature:", feat_imp.iloc[0]["feature"],
      f"({feat_imp.iloc[0]['importance']*100:.1f}%)")

In [ ]:
feat_imp = pd.DataFrame({
    "feature":    X_train.columns.tolist(),
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(9,5))
ax.barh(feat_imp["feature"], feat_imp["importance"],
        color="#2980B9", edgecolor="black")
ax.set_xlabel("Importance Score")
ax.set_title("Feature Importance — What Drives Bus Rush?",
             fontsize=13, fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop feature:", feat_imp.iloc[0]["feature"],
      f"({feat_imp.iloc[0]['importance']*100:.1f}%)")

In [ ]:
import sys
sys.path.append(".")
import predictor as pred_module

# Test prediction with passenger flow
route  = "Guntur-Amaravati"
result = pred_module.predict_rush(
    route, "Monday", 8, "Rainy", False, False
)
print(f"Prediction: {result['emoji']} {result['rush_level']}")
print(f"Confidence: {result['confidence']}%")

flow = pred_module.get_passenger_flow(
    route, "Monday", 8, "Rainy", False, False
)
print(f"\nPassenger Flow — {route}:")
print(f"{'From':25} {'To':25} {'Pax':5} {'%':6} {'Status'}")
print("-" * 70)
for f in flow["flows"]:
    emoji = "🔴" if f["rush"]=="High" \
            else ("🟡" if f["rush"]=="Medium" else "🟢")
    print(f"{f['from_stop']:25} {f['to_stop']:25} "
          f"{f['passengers']:5} {f['percentage']:5}% {emoji}")

print(f"\nMost crowded : {flow['most_crowded']['from_stop']}"
      f" → {flow['most_crowded']['to_stop']}"
      f" ({flow['most_crowded']['passengers']} passengers)")
print(f"Least crowded: {flow['least_crowded']['from_stop']}"
      f" → {flow['least_crowded']['to_stop']}"
      f" ({flow['least_crowded']['passengers']} passengers)")
print(f"\n💡 Smart tip: Board at"
      f" {flow['least_crowded']['from_stop']}"
      f" for a comfortable journey!")

In [3]:
## 7. Conclusions

### Key Findings:
1. **Hour of day** is the strongest predictor (58.5% importance)
2. **Peak hours** (7-10 AM, 5-8 PM) always show High rush
3. **Rainy weather** increases passenger count by ~28%
4. **Weekend trips** are 38% less crowded than weekdays
5. **Amaravati routes** are consistently the busiest

### Passenger Flow Insight:
- The busiest segment is always from the origin stop
- Boarding at intermediate stops gives 30-50% fewer passengers
- Tenali-Vijayawada and Guntur-Amaravati have highest segment loads

### Model Performance:
- Random Forest (80.5%) > Decision Tree (79.9%)
- Most errors between Medium/Low boundary — expected
- Zero data leakage — passenger_count excluded from features

### Next Steps:
- Deploy Flask app on Render.com
- Connect to real APSRTC ETM API for live data
- Add Telugu language support

SyntaxError: invalid character '—' (U+2014) (1347825578.py, line 17)